In [1]:
import pygame; import math

pygame 2.6.1 (SDL 2.28.4, Python 3.12.3)
Hello from the pygame community. https://www.pygame.org/contribute.html


In [2]:
import random as rand

In [3]:
def lerp(A, B, t):
    return A + (B - A) * t

In [4]:
class Level:
    def feed_forward(given_inputs, level):
        for i, given in enumerate(given_inputs):
            level.input_nodes[i] = given

        for i in range(level.output_ct):
            agg = 0
            for j in range(level.input_ct):
                agg += level.input_nodes[j] * level.weights[j][i]
            if agg > level.biases[i]:
                level.output_nodes[i] = 1
            else:
                level.output_nodes[i] = 0

        return level.output_nodes

    def randomize(level):
        for i in range(level.input_ct):
            for j in range(level.output_ct):
                level.weights[i][j] = rand.randint(0, 1) * 2 - 1

        for i in range(len(level.biases)):
            level.biases[i] = rand.randint(0, 1) * 2 - 1

    def __init__(self, input_ct, output_ct):
        self.input_ct = input_ct
        self.output_ct = output_ct

        self.input_nodes = [None] * input_ct
        self.output_nodes = [None] * output_ct

        self.biases = [None] * output_ct
        self.weights = [[0 for _ in range(output_ct)] for _ in range(input_ct)]

        Level.randomize(self)

    def __repr__(self):
        repr = "\n".join(
            [
                f"\tWeights : {self.weights}",
                f"\tBiases : {self.biases}",
                "--------------------------------------------",
            ]
        )
        return repr

In [5]:
class NN:
    def feed_forward(given_inputs, network):
        outputs = Level.feed_forward(given_inputs, network.levels[0])
        for i in range(1, len(network.levels)):
            outputs = Level.feed_forward(outputs, network.levels[i])
        return outputs

    def mutate(network, amount=1):
        for i in range(len(network.levels)):
            # Mutate biases
            for j in range(len(network.levels[i].biases)):
                network.levels[i].biases[j] = lerp(
                    network.levels[i].biases[j], rand.randint(0, 1) * 2 - 1, amount
                )

            # Mutate weights
            for j in range(network.levels[i].input_ct):
                for k in range(network.levels[i].output_ct):
                    network.levels[i].weights[j][k] = lerp(
                        network.levels[i].weights[j][k],
                        rand.randint(0, 1) * 2 - 1,
                        amount,
                    )

    def __init__(self, neuron_ct: list[int]):
        self.levels = []
        for i in range(len(neuron_ct) - 1):
            self.levels.append(Level(neuron_ct[i], neuron_ct[i + 1]))

    def __repr__(self):
        repr = "\n".join([f"Level {i}\n{level}" for i, level in enumerate(self.levels)])
        return repr

In [6]:
class Controller:
    def __init__(self, control_type: str):
        self.forward = False
        self.reverse = False
        self.right = False
        self.left = False

        self.control_type = control_type

    def _keyboard_listeners(self):
        keys = pygame.key.get_pressed()
        self.forward = keys[pygame.K_w]
        self.reverse = keys[pygame.K_s]
        self.left = keys[pygame.K_a]
        self.right = keys[pygame.K_d]

    def update(self):
        if self.control_type == "KEYS":
            self._keyboard_listeners()
        elif self.control_type == "DUMMY":
            self.forward = True

    def __repr__(self):
        return (
            f"\tW : {self.forward}\nA : {self.left}\tS: {self.reverse}\tD: {self.right}"
        )

In [7]:
def get_intersection(A, B, C, D):
    tTop = (D.x - C.x) * (A.y - C.y) - (D.y - C.y)*(A.x - C.x)
    uTop = (C.y - A.y) * (A.x - B.x) - (C.x - A.x)*(A.y - B.y)
    bottom = (D.y - C.y) * (B.x - A.x) - (D.x - C.x)*(B.y-A.y)

    if bottom != 0:
        t = tTop/bottom; u = uTop/bottom
        if t >= 0 and t <= 1 and u >= 0 and u <= 1:
            return {
                "x" : lerp(A.x, B.x, t),
                "y" : lerp(A.y, B.y, t),
                "offset" : t
            }
    return None

In [8]:
def polygon_intersect(poly1, poly2):
    for i in range(len(poly1)):
        for j in range(len(poly2)):
            touch = get_intersection(
                poly1[i],
                poly1[(i + 1) % len(poly1)],
                poly2[j],
                poly2[(j + 1) % len(poly2)],
            )

            if touch is not None:
                return True
            
    return False

In [9]:
class Road:
    def __init__(self, x, width, lane_ct=3):
        self.x = x
        self.width = width
        self.lane_ct = lane_ct

        self.left = x - width / 2
        self.right = x + width / 2

        self.top = 1e9
        self.bottom = -1e9

        self.borders = [
            (
                pygame.Vector2(self.left, self.top),
                pygame.Vector2(self.left, self.bottom),
            ),
            (
                pygame.Vector2(self.right, self.top),
                pygame.Vector2(self.right, self.bottom),
            ),
        ]

    def get_lane_center(self, lane_index):
        return (
            self.left
            + (self.width / self.lane_ct) / 2
            + min(self.lane_ct - 1, lane_index) * (self.width / self.lane_ct)
        )

    def draw(self, screen, camera_offset):
        pygame.draw.rect(
            screen,
            (74, 74, 74),
            (self.left - camera_offset.x, -1000, self.width, 2000),
        )
        pygame.draw.line(
            screen,
            (255, 255, 255),
            pygame.Vector2(self.left, self.top - camera_offset.y),
            pygame.Vector2(self.left, self.bottom - camera_offset.y),
            5,
        )

        # LANES
        for count in range(1, self.lane_ct):
            x = lerp(self.left, self.right, count / self.lane_ct)
            pygame.draw.line(
                screen,
                (
                    (255, 255, 0)
                    if count > 0 and count < self.lane_ct
                    else (255, 255, 255)
                ),
                pygame.Vector2(x, self.top - camera_offset.y),
                pygame.Vector2(x, self.bottom - camera_offset.y),
                5,
            )

        pygame.draw.line(
            screen,
            (255, 255, 255),
            pygame.Vector2(self.right, self.top - camera_offset.y),
            pygame.Vector2(self.right, self.bottom - camera_offset.y),
            5,
        )

In [10]:
class Sensor:
    def __init__(self, car, ray_ct=6):
        self.car = car
        self.ray_ct = ray_ct
        self.ray_length = 100
        self.ray_spread = math.pi

        self.rays = []
        self.readings = []

    def _cast_rays(self):
        self.rays = []
        for ray_i in range(self.ray_ct):
            ray_angle = lerp(
                self.ray_spread / 2,
                -self.ray_spread / 2,
                ray_i / (self.ray_ct - 1) if self.ray_ct > 1 else 0.5,
            ) + math.radians(self.car.angle)
            start = pygame.Vector2(self.car.pos.x, self.car.pos.y)
            end = pygame.Vector2(
                self.car.pos.x - math.sin(ray_angle) * self.ray_length,
                self.car.pos.y - math.cos(ray_angle) * self.ray_length,
            )
            self.rays.append((start, end))

    def _get_reading(self, ray, road_borders, traffic):
        touches = []
        for border in road_borders:
            touch = get_intersection(ray[0], ray[1], border[0], border[1])
            if touch:
                touches.append(touch)
        for traffic_car in traffic:
            polygon = traffic_car.polygon
            for i in range(len(polygon)):
                touch = get_intersection(
                    ray[0], ray[1], polygon[i], polygon[(i + 1) % len(polygon)]
                )
                if touch:
                    touches.append(touch)

        if not touches:
            return None
        else:
            return min(touches, key=lambda t: t["offset"])

    def update(self, road_borders, traffic):
        self._cast_rays()
        self.readings = []
        for ray in self.rays:
            self.readings.append(self._get_reading(ray, road_borders, traffic))

    def draw(self, screen, camera_offset):
        for i, (ray_s, ray_e) in enumerate(self.rays):
            end = ray_e
            if self.readings[i] is not None:
                end = pygame.Vector2(self.readings[i]["x"], self.readings[i]["y"])
            pygame.draw.line(
                screen, (0, 0, 255), ray_s - camera_offset, end - camera_offset, 2
            )
            pygame.draw.line(
                screen, (255, 0, 0), end - camera_offset, ray_e - camera_offset, 2
            )

In [11]:
class Car:
    def __init__(self, pos, width, height, control_type, max_speed):
        self.pos = pygame.Vector2(pos)
        self.width = width
        self.height = height
        self.damaged = False

        # Movement physics
        self.speed = 0
        self.acceleration = 200  # pixels/sec²
        self.friction = 120  # pixels/sec²
        self.max_speed = max_speed
        self.max_reverse_speed = -max_speed

        # Rotation
        self.angle = 0  # degrees
        self.turn_speed = 120  # degrees/sec

        # Polygon
        self.polygon = []
        # Controls
        self.controller = Controller(control_type)

        # Sensor and NN
        self.automatic = True if control_type == "AI" else False
        self.sensor = Sensor(self) if control_type != "DUMMY" else None
        self.NN = NN([self.sensor.ray_ct, 6, 4]) if control_type != "DUMMY" else None

    def _assess_damage(self, road_borders, traffic):
        for border in road_borders:
            if polygon_intersect(self.polygon, border):
                return True

        for traffic_car in traffic:
            if polygon_intersect(self.polygon, traffic_car.polygon):
                return True
        return False

    def _create_polygon(self):
        points = []

        # constant dimensions
        hw = self.width / 2
        hh = self.height / 2

        # variable parameters
        angle = math.radians(self.angle)

        # local corners (relative to center)
        corners = [
            pygame.Vector2(-hw, -hh),  # top-left
            pygame.Vector2(hw, -hh),  # top-right
            pygame.Vector2(hw, hh),  # bottom-right
            pygame.Vector2(-hw, hh),  # bottom-left
        ]

        for corner in corners:
            direction = -1
            rotated = pygame.Vector2(
                corner.x * math.cos(angle) - corner.y * math.sin(angle),
                direction * (corner.x * math.sin(angle) + corner.y * math.cos(angle)),
            )
            points.append(self.pos + rotated)

        return points

    def _move(self, dt):
        # ACCELERATION
        if self.controller.forward:
            self.speed += self.acceleration * dt
        elif self.controller.reverse:
            self.speed -= self.acceleration * dt
        else:
            # Apply friction when no input
            if self.speed > 0:
                self.speed = max(0, self.speed - self.friction * dt)
            elif self.speed < 0:
                self.speed = min(0, self.speed + self.friction * dt)
        # Clamp speed
        self.speed = max(self.max_reverse_speed, min(self.speed, self.max_speed))

        # TURNING
        if self.speed != 0:
            direction = 1 if self.speed > 0 else -1

            if self.controller.left:
                self.angle += self.turn_speed * dt * direction
            if self.controller.right:
                self.angle -= self.turn_speed * dt * direction

        # Final Move
        self.pos.x -= math.sin(math.radians(self.angle)) * self.speed * dt
        self.pos.y -= math.cos(math.radians(self.angle)) * self.speed * dt

    def update(self, dt, road_borders, traffic):

        if not self.damaged:
            # Controller Listens for Keyboard
            self.controller.update()

            # Movement
            self._move(dt)
            self.polygon = self._create_polygon()
            self.damaged = self._assess_damage(road_borders, traffic)

        # Sensor
        if self.sensor is not None:
            self.sensor.update(road_borders, traffic)
            offsets = [1 - t["offset"] if t else 0 for t in self.sensor.readings]

            outputs = NN.feed_forward(offsets, self.NN)

            if self.automatic:
                self.controller.forward = outputs[0]
                self.controller.reverse = outputs[3]
                self.controller.left = outputs[1]
                self.controller.right = outputs[2]

    def draw(self, screen, camera_offset, draw_sensor):

        pygame.draw.polygon(
            screen,
            (
                ((0, 0, 0) if not self.damaged else (0, 255, 0))
                if self.automatic
                else (255, 0, 0)
            ),
            [point - camera_offset for point in self.polygon],
        )

        # Draw sensor
        if self.sensor is not None and draw_sensor:
            self.sensor.draw(screen, camera_offset)

In [12]:
import copy

In [13]:
pygame.init()
pygame.font.init()
font = pygame.font.Font("freesansbold.ttf", 32)
screen = pygame.display.set_mode((1280, 720))
clock = pygame.time.Clock()
running = True
dt = 0

# 100 Simultaneous Generations
N = 100
road = Road(screen.get_width() / 2, 400)


def generate_cars(N):
    return [
        Car((road.get_lane_center(1), screen.get_height() / 2), 30, 40, "AI", 500)
        for _ in range(N)
    ]


cars = generate_cars(N)

traffic = [
    Car((road.get_lane_center(0), screen.get_height() / 2 - 100), 30, 40, "DUMMY", 300),
    Car((road.get_lane_center(1), screen.get_height() / 2 - 80), 30, 40, "DUMMY", 400),
    Car((road.get_lane_center(2), screen.get_height() / 2 - 120), 30, 40, "DUMMY", 200),
    Car((road.get_lane_center(2), screen.get_height() / 2 - 200), 30, 40, "DUMMY", 200),
    Car((road.get_lane_center(1), screen.get_height() / 2 - 180), 30, 40, "DUMMY", 450),
    Car((road.get_lane_center(0), screen.get_height() / 2 - 220), 30, 40, "DUMMY", 300),
]

camera_offset = pygame.Vector2(0, 0)

generation_timer = 0
GEN_LIMIT = 15
gen = 0

mutate_amount = 0.3

text = font.render(f"{gen}", True, (0, 255, 0), (0, 0, 128))
text.get_rect().center = (20, 20)

# Mutating all cars
best_car = cars[0]
best_active_car = cars[0]
best_NN = copy.deepcopy(best_car.NN)
for i in range(N):
    cars[i].NN = copy.deepcopy(best_NN)
    if i != 0:
        NN.mutate(cars[i].NN, mutate_amount)


while running:
    for event in pygame.event.get():
        if event.type == pygame.QUIT:
            running = False
    screen.fill((0, 100, 0))

    # Game Loop

    # Best Car
    active_cars = [c for c in cars if not c.damaged]
    best_active_car = (
        min(active_cars, key=lambda c: c.pos.y) if active_cars else best_active_car
    )
    best_car = min(cars, key=lambda c: c.pos.y)

    if generation_timer >= GEN_LIMIT or not active_cars:
        cars = generate_cars(N)
        best_NN = copy.deepcopy(best_car.NN)
        for i in range(N):
            cars[i].NN = copy.deepcopy(best_NN)
            if i != 0:
                NN.mutate(cars[i].NN, mutate_amount)

        traffic = [
            Car(
                (road.get_lane_center(0), screen.get_height() / 2 - 100),
                30,
                40,
                "DUMMY",
                300,
            ),
            Car(
                (road.get_lane_center(1), screen.get_height() / 2 - 80),
                30,
                40,
                "DUMMY",
                400,
            ),
            Car(
                (road.get_lane_center(2), screen.get_height() / 2 - 120),
                30,
                40,
                "DUMMY",
                200,
            ),
            Car(
                (road.get_lane_center(2), screen.get_height() / 2 - 200),
                30,
                40,
                "DUMMY",
                200,
            ),
            Car(
                (road.get_lane_center(1), screen.get_height() / 2 - 180),
                30,
                40,
                "DUMMY",
                450,
            ),
            Car(
                (road.get_lane_center(0), screen.get_height() / 2 - 220),
                30,
                40,
                "DUMMY",
                300,
            ),
        ]

        generation_timer = 0
        gen += 1
        text = font.render(f"{gen}", True, (0, 255, 0), (0, 0, 128))

    # Camera tracking
    camera_offset.y = best_active_car.pos.y - screen.get_height() / 2
    # Update Traffic
    for traffic_car in traffic:
        traffic_car.update(dt, road.borders, [])

    # Update Generation
    for car in cars:
        car.update(dt, road.borders, traffic)

    # Output
    road.draw(screen, camera_offset)
    for traffic_car in traffic:
        traffic_car.draw(screen, camera_offset, False)
    for car in cars:
        car.draw(screen, camera_offset, False)
    best_car.draw(screen, camera_offset, True)

    screen.blit(text, text.get_rect())

    # Non framerate physics

    dt = clock.tick(120) / 1000
    generation_timer += dt
    pygame.display.flip()
    clock.tick(120)


pygame.font.quit()
pygame.quit()


with open("best_car.txt", "w") as f:
    f.write(f"{best_NN}")

ALSA lib confmisc.c:855:(parse_card) cannot find card '0'
ALSA lib conf.c:5204:(_snd_config_evaluate) function snd_func_card_inum returned error: No such file or directory
ALSA lib confmisc.c:422:(snd_func_concat) error evaluating strings
ALSA lib conf.c:5204:(_snd_config_evaluate) function snd_func_concat returned error: No such file or directory
ALSA lib confmisc.c:1342:(snd_func_refer) error evaluating name
ALSA lib conf.c:5204:(_snd_config_evaluate) function snd_func_refer returned error: No such file or directory
ALSA lib conf.c:5727:(snd_config_expand) Evaluate error: No such file or directory
ALSA lib pcm.c:2721:(snd_pcm_open_noupdate) Unknown PCM default
